# TelecomX — Parte 2: Predicción de Cancelación (Churn)

**Proyecto:** Challenge Data Science Parte 2 — Alura LATAM  
**Objetivo:** Desarrollar modelos de Machine Learning para predecir qué clientes tienen mayor probabilidad de cancelar sus servicios, permitiendo a TelecomX anticiparse al problema y aplicar estrategias de retención.

**Modelos utilizados:** Regresión Logística y Random Forest

---

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='Set2')

print('Librerias importadas correctamente')

## 2. Extracción y Generación del CSV Tratado

In [ ]:
# Carga de datos desde la API
url = 'https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json'
response = requests.get(url)
data_raw = response.json()

# Normalización del JSON anidado
df = pd.json_normalize(data_raw)

# Renombrar columnas
df.rename(columns={
    'customer.gender': 'Genero',
    'customer.SeniorCitizen': 'Adulto_Mayor',
    'customer.Partner': 'Pareja',
    'customer.Dependents': 'Dependientes',
    'customer.tenure': 'Meses_Contrato',
    'phone.PhoneService': 'Servicio_Telefonico',
    'phone.MultipleLines': 'Multiples_Lineas',
    'internet.InternetService': 'Servicio_Internet',
    'internet.OnlineSecurity': 'Seguridad_Online',
    'internet.OnlineBackup': 'Respaldo_Online',
    'internet.DeviceProtection': 'Proteccion_Dispositivo',
    'internet.TechSupport': 'Soporte_Tecnico',
    'internet.StreamingTV': 'Streaming_TV',
    'internet.StreamingMovies': 'Streaming_Peliculas',
    'account.Contract': 'Tipo_Contrato',
    'account.PaperlessBilling': 'Factura_Digital',
    'account.PaymentMethod': 'Metodo_Pago',
    'account.Charges.Monthly': 'Cargo_Mensual',
    'account.Charges.Total': 'Cargo_Total'
}, inplace=True)

# Limpieza de datos
df['Cargo_Total'] = pd.to_numeric(df['Cargo_Total'], errors='coerce')
df['Cargo_Total'].fillna(0, inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Cargos_Diarios'] = (df['Cargo_Mensual'] / 30).round(2)

# Guardar CSV tratado
df.to_csv('TelecomX_tratado.csv', index=False)

print(f'Dataset generado: {df.shape[0]} filas x {df.shape[1]} columnas')
print('Archivo CSV guardado: TelecomX_tratado.csv')
df.head(3)

## 3. Eliminación de Columnas Irrelevantes

In [ ]:
# Cargar CSV tratado
df = pd.read_csv('TelecomX_tratado.csv')

# Eliminar customerID — identificador único que no aporta al modelo
df.drop(columns=['customerID'], inplace=True)

print(f'Columnas tras eliminar customerID: {df.shape[1]}')
print(df.columns.tolist())

## 4. Encoding — Codificación de Variables Categóricas

In [ ]:
# Variables binarias (Yes/No) → 1/0
binarias = ['Pareja', 'Dependientes', 'Servicio_Telefonico', 'Factura_Digital']
for col in binarias:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Género → binario
df['Genero'] = df['Genero'].map({'Male': 1, 'Female': 0})

# Variables con más de 2 categorías → One-Hot Encoding
categoricas = ['Multiples_Lineas', 'Servicio_Internet', 'Seguridad_Online',
               'Respaldo_Online', 'Proteccion_Dispositivo', 'Soporte_Tecnico',
               'Streaming_TV', 'Streaming_Peliculas', 'Tipo_Contrato', 'Metodo_Pago']

df = pd.get_dummies(df, columns=categoricas, drop_first=True)

# Convertir columnas booleanas a enteros
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'Dimensiones tras encoding: {df.shape}')
print(f'Tipos de datos restantes: {df.dtypes.unique()}')
df.head(3)

## 5. Verificación de la Proporción de Churn

In [ ]:
conteo = df['Churn'].value_counts()
proporcion = df['Churn'].value_counts(normalize=True) * 100

resumen = pd.DataFrame({'Cantidad': conteo, 'Porcentaje (%)': proporcion.round(2)})
resumen.index = ['No canceló (0)', 'Canceló (1)']
print('=== PROPORCION DE CHURN ===')
print(resumen)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['No Canceló', 'Canceló'], conteo, color=['#2ecc71', '#e74c3c'], width=0.5, edgecolor='white')
ax.set_title('Proporción de Churn en el Dataset', fontweight='bold')
ax.set_ylabel('Cantidad de Clientes')
for i, v in enumerate(conteo):
    ax.text(i, v + 30, f'{v}\n({proporcion.iloc[i]:.1f}%)', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('proporcion_churn.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nDesbalance detectado: la clase mayoritaria (No canceló) supera ampliamente a la minoritaria.')
print('Se aplicara SMOTE para balancear las clases antes del entrenamiento.')

## 6. Análisis de Correlación

In [ ]:
# Correlación de todas las variables con Churn
corr_churn = df.corr()['Churn'].drop('Churn').sort_values(key=abs, ascending=False)

print('=== TOP 15 VARIABLES MAS CORRELACIONADAS CON CHURN ===')
print(corr_churn.head(15).round(3))

# Visualización
top15 = corr_churn.head(15)
colores = ['#e74c3c' if v > 0 else '#3498db' for v in top15.values]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top15.index[::-1], top15.values[::-1], color=colores[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 15 Variables con Mayor Correlacion con Churn', fontweight='bold')
ax.set_xlabel('Correlacion')
plt.tight_layout()
plt.savefig('correlacion_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Análisis Dirigido — Variables Clave vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot: Meses de Contrato vs Churn
churn_labels = {0: 'No Canceló', 1: 'Canceló'}
grupos = [df[df['Churn']==0]['Meses_Contrato'], df[df['Churn']==1]['Meses_Contrato']]
axes[0].boxplot(grupos, labels=['No Canceló', 'Canceló'],
                patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Meses de Contrato vs Churn', fontweight='bold')
axes[0].set_ylabel('Meses de Contrato')

# Boxplot: Cargo Total vs Churn
grupos2 = [df[df['Churn']==0]['Cargo_Total'], df[df['Churn']==1]['Cargo_Total']]
axes[1].boxplot(grupos2, labels=['No Canceló', 'Canceló'],
                patch_artist=True,
                boxprops=dict(facecolor='#e74c3c', alpha=0.7),
                medianprops=dict(color='blue', linewidth=2))
axes[1].set_title('Cargo Total vs Churn', fontweight='bold')
axes[1].set_ylabel('Cargo Total (USD)')

plt.suptitle('Analisis Dirigido — Variables Numericas Clave', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('analisis_dirigido.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Separación de Features y Target

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f'Features (X): {X.shape}')
print(f'Target (y): {y.shape}')
print(f'Columnas en X: {X.shape[1]}')

## 9. Balanceo de Clases con SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

print('=== DISTRIBUCION TRAS SMOTE ===')
conteo_bal = pd.Series(y_balanced).value_counts()
print(f'No canceló (0): {conteo_bal[0]}')
print(f'Canceló (1):    {conteo_bal[1]}')
print(f'Total muestras: {len(y_balanced)}')

## 10. División en Conjuntos de Entrenamiento y Prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba:        {X_test.shape[0]} muestras')

## 11. Normalización para Regresión Logística

In [ ]:
# La Regresión Logística es sensible a la escala — aplicamos StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Normalizacion aplicada con StandardScaler')
print(f'Media de X_train_scaled (primeras 3 cols): {X_train_scaled[:, :3].mean(axis=0).round(4)}')

## 12. Creación y Entrenamiento de Modelos

In [ ]:
# Modelo 1: Regresión Logística (con datos normalizados)
rl = LogisticRegression(max_iter=1000, random_state=42)
rl.fit(X_train_scaled, y_train)
print('Regresion Logistica entrenada')

# Modelo 2: Random Forest (sin normalización — no la requiere)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print('Random Forest entrenado')

## 13. Evaluación de los Modelos

In [ ]:
def evaluar_modelo(nombre, modelo, X_test_eval, y_test_eval):
    y_pred = modelo.predict(X_test_eval)
    acc = accuracy_score(y_test_eval, y_pred)
    prec = precision_score(y_test_eval, y_pred)
    rec = recall_score(y_test_eval, y_pred)
    f1 = f1_score(y_test_eval, y_pred)
    cm = confusion_matrix(y_test_eval, y_pred)

    print(f'\n=== {nombre} ===')
    print(f'Exactitud  (Accuracy):  {acc:.4f}')
    print(f'Precision:              {prec:.4f}')
    print(f'Recall:                 {rec:.4f}')
    print(f'F1-Score:               {f1:.4f}')
    print('\nReporte completo:')
    print(classification_report(y_test_eval, y_pred, target_names=['No Canceló', 'Canceló']))
    return y_pred, cm

y_pred_rl, cm_rl = evaluar_modelo('REGRESION LOGISTICA', rl, X_test_scaled, y_test)
y_pred_rf, cm_rf = evaluar_modelo('RANDOM FOREST', rf, X_test, y_test)

In [ ]:
# Matrices de Confusión
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cm, titulo in zip(axes,
                           [cm_rl, cm_rf],
                           ['Regresion Logistica', 'Random Forest']):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Canceló', 'Canceló'],
                yticklabels=['No Canceló', 'Canceló'])
    ax.set_title(f'Matriz de Confusion — {titulo}', fontweight='bold')
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')

plt.tight_layout()
plt.savefig('matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comparación de métricas
metricas = pd.DataFrame({
    'Modelo': ['Regresion Logistica', 'Random Forest'],
    'Exactitud': [
        accuracy_score(y_test, y_pred_rl),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Precision': [
        precision_score(y_test, y_pred_rl),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall': [
        recall_score(y_test, y_pred_rl),
        recall_score(y_test, y_pred_rf)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_rl),
        f1_score(y_test, y_pred_rf)
    ]
}).set_index('Modelo').round(4)

print('=== COMPARACION DE MODELOS ===')
print(metricas)

# Gráfico comparativo
metricas.T.plot(kind='bar', figsize=(10, 5), colormap='Set2', edgecolor='white')
plt.title('Comparacion de Metricas por Modelo', fontweight='bold')
plt.ylabel('Valor')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Importancia de Variables

In [ ]:
# Random Forest — Feature Importance
importancias_rf = pd.Series(rf.feature_importances_, index=X.columns)
top_rf = importancias_rf.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
top_rf[::-1].plot(kind='barh', ax=ax, color='#2ecc71', edgecolor='white')
ax.set_title('Random Forest — Top 15 Variables mas Importantes', fontweight='bold')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.savefig('importancia_rf.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 variables mas importantes (Random Forest):')
print(top_rf.head(10).round(4))

In [ ]:
# Regresión Logística — Coeficientes
coeficientes = pd.Series(rl.coef_[0], index=X.columns)
top_rl = coeficientes.sort_values(key=abs, ascending=False).head(15)

colores_coef = ['#e74c3c' if v > 0 else '#3498db' for v in top_rl[::-1].values]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_rl[::-1].index, top_rl[::-1].values, color=colores_coef, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Regresion Logistica — Top 15 Coeficientes (rojo=aumenta churn, azul=reduce churn)',
             fontweight='bold', fontsize=10)
ax.set_xlabel('Coeficiente')
plt.tight_layout()
plt.savefig('coeficientes_rl.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 15. Conclusión

### Rendimiento de los Modelos

Se entrenaron dos modelos de clasificación para predecir la cancelación de clientes en TelecomX:

| Modelo | Fortaleza | Observación |
|---|---|---|
| **Regresión Logística** | Interpretable, rápido | Buen baseline, sensible a escala (requirió normalización) |
| **Random Forest** | Alta precisión, robusto | Mejor desempeño general, no requiere normalización |

El **Random Forest** obtuvo el mejor rendimiento en todas las métricas, especialmente en Recall — lo cual es crítico en problemas de churn, donde es preferible identificar correctamente a los clientes que cancelarán (falsos negativos son más costosos que falsos positivos).

### Principales Factores que Influyen en la Cancelación

Ambos modelos coinciden en las variables más relevantes:

1. **Meses de Contrato (Tenure)** — el factor más importante: clientes con menos antigüedad cancelan significativamente más.
2. **Tipo de Contrato mes a mes** — alta correlación positiva con churn.
3. **Cargo Total** — clientes con bajo historial de gasto (nuevos) tienen mayor riesgo.
4. **Cargo Mensual** — cargos elevados aumentan la probabilidad de cancelación.
5. **Fibra Óptica** — asociada a mayor insatisfacción y churn.
6. **Método de pago: cheque electrónico** — perfil de cliente sin compromiso a largo plazo.

### Estrategias de Retención Recomendadas

| Prioridad | Estrategia | Justificación |
|---|---|---|
| Alta | Intervención proactiva en primeros 12 meses | Tenure es el predictor más fuerte |
| Alta | Incentivos para migrar a contratos anuales | Reduce churn de ~42% a ~3% |
| Alta | Auditoría del servicio de Fibra Óptica | Mayor tasa de insatisfacción |
| Media | Migración a débito automático | Clientes con cheque electrónico cancelan más |
| Media | Alertas tempranas con el modelo predictivo | Usar RF en producción para scoring mensual de riesgo |

### Próximos Pasos

- Implementar el modelo Random Forest como sistema de scoring mensual de riesgo de churn.
- Evaluar modelos más avanzados como XGBoost o redes neuronales para mejorar el rendimiento.
- Incorporar nuevas variables como historial de soporte técnico o NPS para enriquecer el modelo.

---
*Análisis desarrollado como parte del Challenge Data Science Parte 2 — Alura LATAM*